# First model (IRT)


In [ ]:
import polars as pl
import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset

In [ ]:
dataset = load_dataset("najrum/cf-interactions").with_format("polars")


DATA_DIR = "../data"
MODEL_DIR = "../models/irt"

In [ ]:
df = pl.scan_parquet(f"{DATA_DIR}/interactions.parquet")

In [ ]:
user_handles = df.select("user_handle").unique().collect()["user_handle"]
problem_ids = df.select("problem_id").unique().collect()["problem_id"]

user_map = {u: i for i, u in enumerate(user_handles)}
problem_map = {p: i for i, p in enumerate(problem_ids)}

In [ ]:
import pickle

# save maps to disk
with open(f"{MODEL_DIR}/user_map.pkl", "wb") as f:
    pickle.dump(user_map, f)
with open(f"{MODEL_DIR}/problem_map.pkl", "wb") as f:
    pickle.dump(problem_map, f)

In [ ]:
users = pl.DataFrame(
    {"user_handle": list(user_map.keys()), "user_id": list(user_map.values())}
).lazy()
problems = pl.DataFrame(
    {"problem_id": list(problem_map.keys()), "item_id": list(problem_map.values())}
).lazy()
df = (
    df.join(users, on="user_handle", how="left")
    .join(problems, on="problem_id", how="left")
    .with_columns(pl.col("solved").cast(pl.Float32).alias("label"))
    .with_row_index("row_id")
    .select(["row_id", "user_id", "item_id", "label"])
)

In [ ]:
user_train = df.group_by("user_id").agg(pl.col("row_id").first()).select(["row_id"])

item_train = df.group_by("item_id").agg(pl.col("row_id").first()).select(["row_id"])

forced_train_ids = pl.concat([user_train, item_train]).unique()
forced_train_rows = df.join(forced_train_ids, on="row_id", how="inner")
remaining = df.join(forced_train_ids, on="row_id", how="anti")

train_rest = remaining.filter((pl.col("row_id") % 10) < 7)
val_rest = remaining.filter((pl.col("row_id") % 10) == 7)
test_rest = remaining.filter((pl.col("row_id") % 10) >= 8)

pl.concat([forced_train_rows, train_rest]).unique().sink_parquet(
    f"{DATA_DIR}/train.parquet"
)
val_rest.sink_parquet(f"{DATA_DIR}/val.parquet")
test_rest.sink_parquet(f"{DATA_DIR}/test.parquet")

In [ ]:
from torch.utils.data import Dataset, DataLoader


class InteractionDataset(Dataset):
    def __init__(self, path):
        arr = (
            pl.scan_parquet(path)
            .select(["user_id", "item_id", "label"])
            .collect()
            .to_numpy()
        )
        self.users = torch.from_numpy(arr[:, 0].astype("int32"))
        self.items = torch.from_numpy(arr[:, 1].astype("int32"))
        self.labels = torch.from_numpy(arr[:, 2].astype("float32"))

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.labels[idx]


batch_size = 131072
num_workers = 12
train_loader = DataLoader(
    InteractionDataset(f"{DATA_DIR}/train.parquet"),
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
)
val_loader = DataLoader(
    InteractionDataset(f"{DATA_DIR}/val.parquet"),
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
)
test_loader = DataLoader(
    InteractionDataset(f"{DATA_DIR}/test.parquet"),
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
import torch.nn.functional as F


class IRTModel(nn.Module):
    def __init__(self, num_users, num_items):
        super().__init__()
        self.theta = nn.Embedding(num_users, 1)  # user ability
        self.beta = nn.Embedding(num_items, 1)  # item difficulty
        self.alpha = nn.Embedding(num_items, 1)  # item discrimination

        nn.init.normal_(self.theta.weight, std=0.1)
        nn.init.normal_(self.beta.weight, std=0.1)
        nn.init.ones_(self.alpha.weight)

    def forward(self, u, i):
        theta = self.theta(u).squeeze(-1)
        beta = self.beta(i).squeeze(-1)
        alpha = F.softplus(self.alpha(i)).squeeze(-1)  # constrain > 0
        return alpha * (theta - beta)

In [ ]:
def train_one_epoch(model, loader, optimizer, device, scaler):
    model.train()
    pos_weight = torch.tensor([0.3], device=device)
    total_loss = 0
    for u, i, label in loader:
        u = u.to(device, non_blocking=True)
        i = i.to(device, non_blocking=True)
        label = label.to(device, non_blocking=True)
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):  # type: ignore
            pred = model(u, i)
            loss = F.binary_cross_entropy_with_logits(
                pred, label, pos_weight=pos_weight
            )

            centering_penalty = 0.01 * (
                model.theta.weight.mean() ** 2 + model.beta.weight.mean() ** 2
            )
            loss += centering_penalty

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
from sklearn.metrics import roc_auc_score


def evaluate(model, loader, device):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for user, item, label in loader:
            user = user.to(device, non_blocking=True)
            item = item.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                pred = model(user, item)
            preds.append(pred.cpu().numpy())
            targets.append(label.numpy())
    preds = np.concatenate(preds)
    targets = np.concatenate(targets)
    return roc_auc_score(targets, preds)

In [ ]:
model = IRTModel(num_users=len(user_map), num_items=len(problem_map)).to(device)

In [ ]:
import time

optimizer = torch.optim.AdamW(
    [
        {"params": model.theta.weight, "lr": 1e-1},  # High LR for users
        {"params": model.beta.weight, "lr": 1e-2},  # Medium LR for items
        {"params": model.alpha.weight, "lr": 1e-3},
    ],
    weight_decay=0,
)
scaler = torch.amp.GradScaler("cuda")  # type: ignore
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=1, factor=0.5
)

patience = 5
best_auc, no_improve = 0, 0
prev_time = time.time()

for epoch in range(50):
    train_loss = train_one_epoch(model, train_loader, optimizer, device, scaler)
    val_auc = evaluate(model, val_loader, device)

    scheduler.step(val_auc)

    print(f"Epoch {epoch+1} | Loss {train_loss:.4f} | AUC {val_auc:.4f}")

    if val_auc > best_auc + 1e-4:
        best_auc, no_improve = val_auc, 0
        torch.save(model.state_dict(), f"{MODEL_DIR}/model.pt")
    else:
        no_improve += 1
        if no_improve >= patience:
            break
print("Done training.")

In [ ]:
model.load_state_dict(torch.load(f"{MODEL_DIR}/model.pt", weights_only=True))

In [ ]:
test_auc = evaluate(model, test_loader, device)
print(f"Test AUC {test_auc:.4f}")

In [ ]:
handle = "TomatoPotato"
problem_id = "2218.A"

In [ ]:
total = len(test_loader.dataset)
solved = sum(test_loader.dataset.labels.numpy())
print(
    f"Solved: {solved} ({solved/total:.2%}), Not Solved: {total-solved} ({(total-solved)/total:.2%})"
)

user_id = user_map[handle]
user_rows = df.filter(pl.col("user_id") == user_id).collect()
user_solved = user_rows.filter(pl.col("label") == 1).shape[0]
user_not_solved = user_rows.filter(pl.col("label") == 0).shape[0]
print(f"{handle} solved: {user_solved}, not solved: {user_not_solved}")

In [ ]:
model.theta(torch.tensor([user_map[handle]], device=device)).item(), model.beta(
    torch.tensor([problem_map[problem_id]], device=device)
).item(), model.alpha(torch.tensor([problem_map[problem_id]], device=device)).item()

In [ ]:
pred = model(
    torch.tensor([user_map[handle]], device=device),
    torch.tensor([problem_map[problem_id]], device=device),
)
prob = torch.sigmoid(pred)
print(f"Predicted probability that {handle} solves {problem_id}: {prob.item():.4f}")